In [7]:
from pathlib import Path
import pandas as pd
import numpy as np

In [8]:
NULL_STRING_PATTERNS = {'NULL', 'null', 'N/A', 'NA', 'n/a', 'na', '-'}
DATA = Path("Dataset")
P_DATA = Path("Processed_Dataset")
CHUNK = 500_000

## Công cụ Profiling Dữ liệu - `summarize_df()`

Công cụ profiling pandas DataFrame toàn diện với phát hiện bất thường và quy tắc nghiệp vụ.

### Tính năng:
- **Thống kê cơ bản**: dtype, giá trị thiếu, trống thật, chuỗi NULL/null, tính độc nhất
- **Phân tích chuỗi**: độ dài, khoảng trắng đầu/cuối
- **Phân tích số**: min, max, mean, median, std, outlier theo IQR
- **Quy tắc nghiệp vụ**: hàm kiểm tra tùy chỉnh
- **Chất lượng dữ liệu**: cờ bất thường, mức độ nghiêm trọng
- **Xuất**: xuất CSV, hiển thị styled dataframe


# Kiểm tra dữ liệu FMCG Sales

Notebook đọc lần lượt các CSV trong `Dataset/`, báo cáo kích thước, missing values,...

In [11]:
assert DATA.is_dir(), f"Không tìm thấy thư mục {DATA.resolve()}"
assert P_DATA.is_dir(), f"Không tìm thấy thư mục {P_DATA.resolve()}"

In [12]:
def summarize_df(
    df: pd.DataFrame,
    name: str = "unname",
    primary_key: list = None,
    custom_validators: dict = None,
    export_csv: str = None,
    styled: bool = False,
) -> pd.DataFrame:
    r"""
    Công cụ profiling pandas DataFrame toàn diện.
    
    Phân tích từng cột và trả về bảng tóm tắt chi tiết với:
    - Thông tin cơ bản (dtype, row_count)
    - Giá trị thiếu (count), trống thật, chuỗi NULL/null/N/A/...
    - Tính độc nhất (unique_count)
    - Phân tích chuỗi (độ dài, khoảng trắng đầu/cuối)
    - Phân tích số (min, max, mean, median, std, outliers)
    - Kiểm tra quan hệ (one-to-many anomalies)
    - Kiểm tra trùng lặp
    - Kiểm tra quy tắc nghiệp vụ (custom validators)
    - Cờ bất thường và mức độ nghiêm trọng
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame đầu vào cần profiling
    name: str, optional
        Tên dùng để gọi DataFrame
    primary_key : list, optional
        Danh sách cột dùng để kiểm tra khóa chính/trùng lặp
        Ví dụ: ['CountryID', 'Year']
    custom_validators : dict, optional
        {column_name: callable} cho các quy tắc nghiệp vụ
        Ví dụ: {'Age': lambda x: (x >= 0) & (x <= 150)}
    export_csv : str, optional
        Đường dẫn file CSV để xuất kết quả
    styled : bool
        Trả về styled dataframe nếu True
    
    Returns:
    --------
    pd.DataFrame
        Bảng tóm tắt với mỗi dòng tương ứng một cột, index là tên cột
    """
    
    
    rows = []
    n_rows = len(df)
    
    def count_blank(series):
        """Chuỗi trống thật: rỗng hoặc chỉ khoảng trắng (không tính NaN)."""
        if series.dtype == 'object':
            def is_blank(val):
                if pd.isna(val):
                    return False
                s = str(val)
                return s == '' or s.strip() == ''
            return int(series.apply(is_blank).sum())
        return 0
    
    def count_null_strings(series):
        """Chuỗi mang nghĩa null: NULL, null, N/A, NA, -, ..."""
        if series.dtype == 'object':
            str_series = series.astype(str)
            return sum((str_series == pattern).sum() for pattern in NULL_STRING_PATTERNS)
        return 0
    
    # Hàm phụ: Đếm khoảng trắng đầu/cuối trong cột chuỗi
    def count_leading_trailing_spaces(series):
        if series.dtype == 'object':
            str_series = series.astype(str)
            non_na = str_series[series.notna()]
            if len(non_na) > 0:
                has_space = (non_na.str.len() > non_na.str.strip().str.len()).sum()
                return int(has_space)
        return 0
    
    # Vòng lặp chính phân tích từng cột
    for col in df.columns:
        series = df[col]
        dtype = str(series.dtype)
        
        # Thông tin cơ bản
        missing_count = int(series.isna().sum())
        
        row = {
            'column': col,
            'dtype': dtype,
            'row_count': n_rows,
            'missing_count': missing_count,
            'blank_count': count_blank(series),
            'null_string_count': count_null_strings(series),
        }
        
        # Tính độc nhất
        row['unique_count'] = int(series.nunique())
        
        # Phân tích chuỗi (dtype object)
        if dtype == 'object':
            str_series = series.astype(str)
            non_na = str_series[series.notna()]
            
            if len(non_na) > 0:
                lengths = non_na.str.len()
                row['min_length'] = int(lengths.min())
                row['max_length'] = int(lengths.max())
                row['avg_length'] = round(lengths.mean(), 2)
            else:
                row['min_length'] = None
                row['max_length'] = None
                row['avg_length'] = None
            
            row['leading_trailing_spaces'] = count_leading_trailing_spaces(series)
        else:
            row['min_length'] = None
            row['max_length'] = None
            row['avg_length'] = None
            row['leading_trailing_spaces'] = 0
        
        # Phân tích số (int64, float64)
        is_id_column = col.upper().endswith('ID')

        if pd.api.types.is_numeric_dtype(series) and not is_id_column:
            numeric_series = pd.to_numeric(series, errors='coerce')
            valid_numeric = numeric_series.dropna()
            
            if len(valid_numeric) > 0:
                row['min'] = round(valid_numeric.min(), 4)
                row['max'] = round(valid_numeric.max(), 4)
                row['mean'] = round(valid_numeric.mean(), 4)
                row['median'] = round(valid_numeric.median(), 4)
                row['std'] = round(valid_numeric.std(), 4)
                
                # Phát hiện outlier theo IQR
                Q1 = valid_numeric.quantile(0.25)
                Q3 = valid_numeric.quantile(0.75)
                IQR = Q3 - Q1
                outliers = ((valid_numeric < (Q1 - 1.5 * IQR)) | (valid_numeric > (Q3 + 1.5 * IQR))).sum()
                row['outlier_count'] = int(outliers)
            else:
                row['min'] = None
                row['max'] = None
                row['mean'] = None
                row['median'] = None
                row['std'] = None
                row['outlier_count'] = 0
        else:
            row['min'] = None
            row['max'] = None
            row['mean'] = None
            row['median'] = None
            row['std'] = None
            row['outlier_count'] = 0
        
        # Validator tùy chỉnh (quy tắc nghiệp vụ)
        if custom_validators and col in custom_validators:
            try:
                validator_func = custom_validators[col]
                valid_mask = series.notna() & validator_func(series)
                invalid_count = int((~valid_mask).sum() - missing_count)
                row['custom_validator_invalid'] = invalid_count
            except:
                row['custom_validator_invalid'] = None
        else:
            row['custom_validator_invalid'] = None
        
        rows.append(row)
    
    # Tạo DataFrame tóm tắt
    summary = pd.DataFrame(rows).set_index('column')
    
    # Kiểm tra khóa chính/trùng lặp
    if primary_key and all(k in df.columns for k in primary_key):
        summary['duplicated_key_count'] = 0
        dup_mask = df.duplicated(subset=primary_key, keep=False)
        dup_count = int(dup_mask.sum())
        # Gán kết quả vào cột khóa chính đầu tiên để dễ quan sát
        if len(primary_key) > 0 and primary_key[0] in summary.index:
            summary.loc[primary_key[0], 'duplicated_key_count'] = dup_count
    
    # Cờ bất thường: đánh dấu cột có vấn đề
    def is_anomaly(row):
        issues = 0
        if n_rows > 0 and row.get('missing_count', 0) > n_rows * 0.5:
            issues += 1
        if row.get('blank_count', 0) > 0 or row.get('null_string_count', 0) > 0:
            issues += 1
        if pd.notna(row.get('custom_validator_invalid', 0)) and row['custom_validator_invalid'] > 0:
            issues += 1
        if pd.notna(row.get('outlier_count', 0)) and row['outlier_count'] > (n_rows * 0.05):
            issues += 1
        if pd.notna(row.get('leading_trailing_spaces', 0)) and row['leading_trailing_spaces'] > 0:
            issues += 1
        return 'Yes' if issues > 0 else 'No'
    
    summary['has_anomaly'] = summary.apply(is_anomaly, axis=1)
    
    # Mức độ nghiêm trọng
    def severity_level(row):
        severity = 'Low'
        if row['has_anomaly'] == 'No':
            return 'None'
        if n_rows > 0 and row.get('missing_count', 0) > n_rows * 0.8:
            severity = 'Critical'
        elif n_rows > 0 and row.get('missing_count', 0) > n_rows * 0.5:
            severity = 'High'
        elif row.get('blank_count', 0) > 0 or row.get('null_string_count', 0) > 0:
            severity = 'Medium'
        return severity
    
    summary['severity_level'] = summary.apply(severity_level, axis=1)
    
    # In tiêu đề
    header = f"\n{'=' * 100}\n  DataFrame: {name} |  shape: {df.shape}\n{'=' * 100}"
    print(header)
    
    # Xuất CSV nếu được yêu cầu
    if export_csv:
        summary.to_csv(export_csv)
        print(f"\nSummary exported to: {export_csv}")
    
    # Trả về styled dataframe hoặc dataframe bình thường
    if styled:
        def highlight_anomalies(val):
            if isinstance(val, str) and val == 'Yes':
                return 'background-color: #ffcccc'
            elif isinstance(val, str) and val in ['High', 'Critical']:
                return 'background-color: #ff9999'
            elif isinstance(val, str) and val == 'Medium':
                return 'background-color: #ffeb99'
            return ''
        
        return summary.style.applymap(highlight_anomalies, subset=['has_anomaly', 'severity_level'])
    
    return summary

## Output Columns Reference

| Column | Type | Description |
|--------|------|-------------|
| **dtype** | str | Data type (int64, float64, object) |
| **row_count** | int | Total rows in dataframe |
| **missing_count** | int | NaN/None values |
| **blank_count** | int | Empty string or whitespace-only (not NaN) |
| **null_string_count** | int | Literal strings: NULL, null, N/A, NA, n/a, na, - |
| **unique_count** | int | Distinct values |
| **min_length, max_length, avg_length** | float | String column length statistics |
| **leading_trailing_spaces** | int | Count of values with leading/trailing spaces |
| **min, max, mean, median, std** | float | Numeric statistics |
| **outlier_count** | int | Values beyond 1.5×IQR from Q1/Q3 |
| **custom_validator_invalid** | int | Business rule violations |
| **duplicated_key_count** | int | Duplicate primary key violations |
| **has_anomaly** | str | 'Yes'/'No' - has any data quality issue |
| **severity_level** | str | 'None', 'Low', 'Medium', 'High', 'Critical'

## 1. Bảng dimension (đọc toàn bộ)

### 1.1. Categories, Countries, Cities, Products, Employees, Customers

In [13]:
categories = pd.read_csv(DATA / "categories.csv", keep_default_na=False)
countries = pd.read_csv(DATA / "countries.csv", keep_default_na=False)
cities = pd.read_csv(DATA / "cities.csv", dtype={"Zipcode": str}, keep_default_na=False)
products = pd.read_csv(DATA / "products.csv", keep_default_na=False)
employees = pd.read_csv(DATA / "employees.csv", keep_default_na=False)
customers = pd.read_csv(DATA / "customers.csv", keep_default_na=False)

for name, df in [
    ("categories", categories),
    ("countries", countries),
    ("cities", cities),
    ("products", products),
    ("employees", employees),
    ("customers", customers),
]:
    summary = summarize_df(df=df, name=name)
    display(summary)


  DataFrame: categories |  shape: (11, 2)


,dtype,row_count,missing_count,blank_count,null_string_count,unique_count,min_length,max_length,avg_length,leading_trailing_spaces,min,max,mean,median,std,outlier_count,custom_validator_invalid,has_anomaly,severity_level
column,,,,,,,,,,,,,,,,,,,
CategoryID,int64,11,0,0,0,11,NaN,NaN,NaN,0,None,None,None,None,None,0,None,No,None
CategoryName,object,11,0,0,0,11,4.0,11.0,7.09,0,None,None,None,None,None,0,None,No,None



  DataFrame: countries |  shape: (206, 3)


,dtype,row_count,missing_count,blank_count,null_string_count,unique_count,min_length,max_length,avg_length,leading_trailing_spaces,min,max,mean,median,std,outlier_count,custom_validator_invalid,has_anomaly,severity_level
column,,,,,,,,,,,,,,,,,,,
CountryID,int64,206,0,0,0,206,NaN,NaN,NaN,0,None,None,None,None,None,0,None,No,None
CountryName,object,206,0,0,0,206,4.0,14.0,7.78,0,None,None,None,None,None,0,None,No,None
CountryCode,object,206,0,0,1,206,2.0,2.0,2.00,0,None,None,None,None,None,0,None,Yes,Medium



  DataFrame: cities |  shape: (96, 4)


...

,dtype,row_count,missing_count,blank_count,null_string_count,unique_count,min_length,max_length,avg_length,leading_trailing_spaces,min,max,mean,median,std,outlier_count,custom_validator_invalid,has_anomaly,severity_level
column,,,,,,,,,,,,,,,,,,,
ProductID,int64,452,0,0,0,452,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,0,None,No,None
ProductName,object,452,0,0,0,452,4.0,33.0,20.41,0,NaN,NaN,NaN,NaN,NaN,0,None,No,None
Price,float64,452,0,0,0,452,NaN,NaN,NaN,0,0.0449,99.8755,50.8015,52.4995,28.6167,0,None,No,None
CategoryID,int64,452,0,0,0,11,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,0,None,No,None
Class,object,452,0,0,0,3,3.0,6.0,4.35,0,NaN,NaN,NaN,NaN,NaN,0,None,No,None
ModifyDate,object,452,0,0,0,452,23.0,23.0,23.00,0,NaN,NaN,NaN,NaN,NaN,0,None,No,None
Resistant,object,452,0,0,0,3,4.0,7.0,6.02,0,NaN,NaN,NaN,NaN,NaN,0,None,No,None
IsAllergic,object,452,0,0,0,3,4.0,7.0,5.23,0,NaN,NaN,NaN,NaN,NaN,0,None,No,None
VitalityDays,float64,452,0,0,0,85,NaN,NaN,NaN,0,0.0000,120.0000,26.0310,0.0000,39.0612,0,None,No,None



  DataFrame: employees |  shape: (23, 8)


,dtype,row_count,missing_count,blank_count,null_string_count,unique_count,min_length,max_length,avg_length,leading_trailing_spaces,min,max,mean,median,std,outlier_count,custom_validator_invalid,has_anomaly,severity_level
column,,,,,,,,,,,,,,,,,,,
EmployeeID,int64,23,0,0,0,23,NaN,NaN,NaN,0,None,None,None,None,None,0,None,No,None
FirstName,object,23,0,0,0,22,4.0,9.0,5.87,0,None,None,None,None,None,0,None,No,None
MiddleInitial,object,23,0,0,0,14,1.0,1.0,1.00,0,None,None,None,None,None,0,None,No,None
LastName,object,23,0,0,0,23,4.0,9.0,5.87,0,None,None,None,None,None,0,None,No,None
BirthDate,object,23,0,0,0,23,23.0,23.0,23.00,0,None,None,None,None,None,0,None,No,None
Gender,object,23,0,0,0,2,1.0,1.0,1.00,0,None,None,None,None,None,0,None,No,None
CityID,int64,23,0,0,0,18,NaN,NaN,NaN,0,None,None,None,None,None,0,None,No,None
HireDate,object,23,0,0,0,23,23.0,23.0,23.00,0,None,None,None,None,None,0,None,No,None



  DataFrame: customers |  shape: (98759, 6)


,dtype,row_count,missing_count,blank_count,null_string_count,unique_count,min_length,max_length,avg_length,leading_trailing_spaces,min,max,mean,median,std,outlier_count,custom_validator_invalid,has_anomaly,severity_level
column,,,,,,,,,,,,,,,,,,,
CustomerID,int64,98759,0,0,0,98759,NaN,NaN,NaN,0,None,None,None,None,None,0,None,No,None
FirstName,object,98759,0,0,0,969,2.0,9.0,5.79,0,None,None,None,None,None,0,None,No,None
MiddleInitial,object,98759,0,0,977,27,1.0,4.0,1.03,0,None,None,None,None,None,0,None,Yes,Medium
LastName,object,98759,0,0,0,995,2.0,10.0,6.09,0,None,None,None,None,None,0,None,No,None
CityID,int64,98759,0,0,0,96,NaN,NaN,NaN,0,None,None,None,None,None,0,None,No,None
Address,object,98759,0,0,0,80134,10.0,35.0,19.22,0,None,None,None,None,None,0,None,No,None


* Table countries: missing 1 value tại cột CountryCode. Tuy nhiên vì không ảnh hưởng đến giá trị dữ liệu nên tạm thời bỏ qua.
* Table customers: missing 977 values tại cột MiddleInital. Tức ở đây có 977 người không có tên đệm trong tên. Không ảnh hưởng.

### 1.2. Sales (đọc theo chunk — ~6.7M dòng)

In [ ]:
sales_parts = []

for i, chunk in enumerate(
    pd.read_csv(DATA / "sales.csv", chunksize=CHUNK, parse_dates=["SalesDate"])
):
    sales_parts.append(chunk)
    print(f"  chunk {i + 1}: {len(chunk):,} dòng")

sales = pd.concat(sales_parts, ignore_index=True)
del sales_parts
print(f"\nTổng sales: {sales.shape[0]:,} dòng x {sales.shape[1]} cột")
sales_summary = summarize_df(sales, name="sales")
display(sales_summary)

  chunk 1: 500,000 dòng
  chunk 2: 500,000 dòng
  chunk 3: 500,000 dòng
  chunk 4: 500,000 dòng
  chunk 5: 500,000 dòng
  chunk 6: 500,000 dòng
  chunk 7: 500,000 dòng
  chunk 8: 500,000 dòng
  chunk 9: 500,000 dòng
  chunk 10: 500,000 dòng
  chunk 11: 500,000 dòng
  chunk 12: 500,000 dòng
  chunk 13: 500,000 dòng
  chunk 14: 258,125 dòng

Tổng sales: 6,758,125 dòng x 9 cột

  DataFrame: sales |  shape: (6758125, 9)


,dtype,row_count,missing_count,blank_count,null_string_count,unique_count,min_length,max_length,avg_length,leading_trailing_spaces,min,max,mean,median,std,outlier_count,custom_validator_invalid,has_anomaly,severity_level
column,,,,,,,,,,,,,,,,,,,
SalesID,int64,6758125,0,0,0,6758125,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,0,None,No,None
SalesPersonID,int64,6758125,0,0,0,23,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,0,None,No,None
CustomerID,int64,6758125,0,0,0,98759,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,0,None,No,None
ProductID,int64,6758125,0,0,0,452,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,0,None,No,None
Quantity,int64,6758125,0,0,0,25,NaN,NaN,NaN,0,1.0,25.0,13.004,13.0,7.2097,0,None,No,None
Discount,float64,6758125,0,0,0,3,NaN,NaN,NaN,0,0.0,0.2,0.030,0.0,0.0640,1351194,None,Yes,Low
TotalPrice,float64,6758125,0,0,0,1,NaN,NaN,NaN,0,0.0,0.0,0.000,0.0,0.0000,0,None,No,None
SalesDate,datetime64[ns],6758125,67526,0,0,6670068,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,0,None,No,None
TransactionNumber,object,6758125,0,0,0,6758125,20.0,20.0,20.0,0,NaN,NaN,NaN,NaN,NaN,0,None,No,None


In [12]:
df_missing = sales[sales["SalesDate"].isna()]
display(df_missing.head(20))

,SalesID,SalesPersonID,CustomerID,ProductID,Quantity,Discount,TotalPrice,SalesDate,TransactionNumber
50,51,21,12478,215,4,0.2,0.0,NaT,H8B08JBXS1TOWBJH3XMS
227,228,18,70321,74,18,0.0,0.0,NaT,SJ5VWMCZHIC2XBUEHEZJ
295,296,21,71322,219,19,0.0,0.0,NaT,8IXG8EPHL6LSLY3264G9
347,348,9,92737,352,24,0.0,0.0,NaT,HXGMYR6J7HWJUOTKB19Q
355,356,18,48857,278,13,0.0,0.0,NaT,0KLH8NGRQAGYRCVM4IF8
362,363,18,77154,133,20,0.1,0.0,NaT,FQUXHKQGNRYKRCU50IWY
480,481,1,43991,23,12,0.2,0.0,NaT,R8H5HEWDNSXXGYQ9XL3H
502,503,23,27217,331,7,0.0,0.0,NaT,6G0DTZL5Y4NAP1GNAUG8
578,579,23,54690,415,14,0.0,0.0,NaT,C5CJL7I4JMNKHG2X557Q
692,693,1,86443,308,22,0.0,0.0,NaT,4IZML962H01TJ42Q3GK3


## 2. Kiểm tra khóa & quan hệ dimension

In [25]:
# =========================================================
# Dimension & Relationship Checks
# =========================================================

checks = []

def add_check(table, column, check_name, invalid_count):
    checks.append({
        "Table": table,
        "Column": column,
        "Check": check_name,
        "InvalidCount": int(invalid_count),
        "OK": invalid_count == 0
    })

# =========================================================
# 1. Surrogate Key Checks
# =========================================================

for name, df, col in [
    ("categories", categories, "CategoryID"),
    ("countries", countries, "CountryID"),
    ("cities", cities, "CityID"),
    ("products", products, "ProductID"),
    ("employees", employees, "EmployeeID"),
    ("customers", customers, "CustomerID"),
    ("sales", sales, "SalesID"), 
]:

    ids = sorted(df[col].dropna().unique())
    expected = list(range(1, len(ids) + 1))

    invalid_count = 0 if ids == expected else abs(len(expected) - len(ids))

    add_check(
        table=name,
        column=col,
        check_name="Surrogate key continuous 1..n",
        invalid_count=invalid_count
    )

# =========================================================
# 2. Foreign Key Checks
# =========================================================

# cities -> countries
invalid_country = ~cities["CountryID"].isin(countries["CountryID"])

add_check(
    table="cities",
    column="CountryID",
    check_name="FK -> countries.CountryID",
    invalid_count=invalid_country.sum()
)

# products -> categories
invalid_cat = ~products["CategoryID"].isin(categories["CategoryID"])

add_check(
    table="products",
    column="CategoryID",
    check_name="FK -> categories.CategoryID",
    invalid_count=invalid_cat.sum()
)

# customers / employees -> cities
for name, df in [
    ("customers", customers),
    ("employees", employees)
]:

    invalid_city = ~df["CityID"].isin(cities["CityID"])

    add_check(
        table=name,
        column="CityID",
        check_name="FK -> cities.CityID",
        invalid_count=invalid_city.sum()
    )

# sales -> employees / customers / products
for col, valid_ids, target_table in [
    ("SalesPersonID", employees["EmployeeID"], "employees"),
    ("CustomerID", customers["CustomerID"], "customers"),
    ("ProductID", products["ProductID"], "products"),
]:

    invalid_fk = ~sales[col].isin(valid_ids)

    add_check(
        table="sales",
        column=col,
        check_name=f"FK -> {target_table}.{col}",
        invalid_count=invalid_fk.sum()
    )

# =========================================================
# Result
# =========================================================

check_df = pd.DataFrame(checks)

print("\n" + "=" * 80)
print("Dimension & Relationship Checks")
print("=" * 80)

display(check_df)


Dimension & Relationship Checks


,Table,Column,Check,InvalidCount,OK
0,categories,CategoryID,Surrogate key continuous 1..n,0,True
1,countries,CountryID,Surrogate key continuous 1..n,0,True
2,cities,CityID,Surrogate key continuous 1..n,0,True
3,products,ProductID,Surrogate key continuous 1..n,0,True
4,employees,EmployeeID,Surrogate key continuous 1..n,0,True
5,customers,CustomerID,Surrogate key continuous 1..n,0,True
6,sales,SalesID,Surrogate key continuous 1..n,0,True
7,cities,CountryID,FK -> countries.CountryID,0,True
8,products,CategoryID,FK -> categories.CategoryID,0,True
9,customers,CityID,FK -> cities.CityID,0,True


## 3. Lưu lại các files

### 3.1. Sửa các NULL, NaN,...

In [14]:
customers["MiddleInitial"] = customers["MiddleInitial"].replace(
    list(NULL_STRING_PATTERNS),
    np.nan
)

customers.to_csv(
        P_DATA / "customers.csv",
        index=False,
        header=True
    )

### 3.2. Tính lại doanh thu (TotalPrice trong file đang = 0)

In [ ]:
# ===== Load products =====
products_ref = products[["ProductID", "Price"]]


for i, chunk in enumerate(
    pd.read_csv(DATA / "sales.csv", chunksize=CHUNK, parse_dates=["SalesDate"])
):

    # Merge
    sales_rev = chunk.merge(
        products_ref,
        on="ProductID",
        how="left"
    )

    # Tính TotalPrice
    sales_rev["TotalPrice"] = (
        sales_rev["Quantity"]
        * sales_rev["Price"]
        * (1 - sales_rev["Discount"])
    )

    # Ghi file
    sales_rev.to_csv(
        P_DATA / "sales.csv",
        mode="a",
        index=False,
        header=(i == 0)
    )

    print(f"Done chunk {i+1}")

print("Finished!")


Done chunk 1
Done chunk 2
Done chunk 3
Done chunk 4
Done chunk 5
Done chunk 6
Done chunk 7
Done chunk 8
Done chunk 9
Done chunk 10
Done chunk 11
Done chunk 12
Done chunk 13
Done chunk 14
Finished!
